# 18 · Databases with Python

Pipelines read from and write to databases constantly. Python's **DB-API** gives
every database a common interface; `sqlite3` ships in the standard library so we
can practice with zero setup. We'll also meet **SQLAlchemy Core**, the toolkit
most data pipelines actually use.

In [ ]:
# ▶ Run this first. Locates the sample data no matter where the kernel starts.
from pathlib import Path

def find_data() -> Path:
    here = Path.cwd()
    for base in (here, *here.parents):
        if (base / 'data' / 'raw').exists():
            return base / 'data'
    raise FileNotFoundError('Run: uv run python data/build_data.py')

DATA = find_data()
RAW = DATA / 'raw'
print('Data directory:', DATA)
print('Raw files:', sorted(p.name for p in RAW.glob('*')))

## Connect, cursor, query (the DB-API)

The pattern is universal: `connect()` → `cursor()` → `execute()` →
`fetchall()`. It's identical for Postgres/MySQL/etc., just a different driver
and connection string.

In [ ]:
import sqlite3

con = sqlite3.connect(DATA / 'retail.db')
cur = con.cursor()
cur.execute('SELECT status, COUNT(*), ROUND(SUM(amount), 2) '
            'FROM orders GROUP BY status')
for row in cur.fetchall():
    print(row)
con.close()

## Parameterized queries — never format SQL by hand

Building SQL with f-strings invites **SQL injection** and breaks on quotes. Pass
values as parameters with `?` placeholders (SQLite) — the driver escapes them
safely.

In [ ]:
import sqlite3
con = sqlite3.connect(DATA / 'retail.db')
cur = con.cursor()

country = 'US'
min_amount = 100
cur.execute(
    'SELECT o.order_id, o.amount FROM orders o '
    'JOIN customers c ON c.customer_id = o.customer_id '
    'WHERE c.country = ? AND o.amount >= ? LIMIT 5',
    (country, min_amount),        # parameters as a tuple
)
for row in cur.fetchall():
    print(row)
con.close()

## Rows as dicts

By default rows are tuples. Set a `row_factory` to get name-based access — much
safer than remembering column positions.

In [ ]:
import sqlite3
con = sqlite3.connect(DATA / 'retail.db')
con.row_factory = sqlite3.Row          # rows behave like dicts
cur = con.cursor()
cur.execute('SELECT customer_id, name, country FROM customers LIMIT 3')
for row in cur.fetchall():
    print(row['customer_id'], row['name'], '->', row['country'])
con.close()

## Writing data: INSERT, transactions, bulk load

Use `executemany` for bulk inserts and **commit** to persist. Wrapping writes in
a transaction means either all rows land or none do — critical for correct
loads. Here we build a small aggregate table.

In [ ]:
import sqlite3
con = sqlite3.connect(DATA / 'retail.db')
cur = con.cursor()

cur.execute('DROP TABLE IF EXISTS revenue_by_country')
cur.execute('CREATE TABLE revenue_by_country (country TEXT, revenue REAL)')

rows = cur.execute(
    'SELECT UPPER(c.country) AS country, ROUND(SUM(o.amount), 2) '
    'FROM orders o JOIN customers c ON c.customer_id = o.customer_id '
    "WHERE o.status = 'completed' GROUP BY UPPER(c.country)"
).fetchall()

cur.executemany('INSERT INTO revenue_by_country VALUES (?, ?)', rows)
con.commit()                          # persist the transaction

for r in cur.execute('SELECT * FROM revenue_by_country ORDER BY revenue DESC LIMIT 5'):
    print(r)
con.close()

## SQLAlchemy Core

Most Python data tools speak **SQLAlchemy**. An `engine` abstracts the database
behind one connection string, so the same code targets SQLite in dev and
Postgres in prod. `text()` runs SQL with named `:params`, and it integrates
directly with pandas (covered in the companion pandas-numpy bootcamp).

In [ ]:
from sqlalchemy import create_engine, text

engine = create_engine(f'sqlite:///{DATA / "retail.db"}')
with engine.connect() as conn:
    result = conn.execute(
        text('SELECT category, COUNT(*) AS n FROM products '
             'GROUP BY category ORDER BY n DESC'),
    )
    for row in result:
        print(row.category, row.n)     # attribute access by column name

### Recap

DB-API pattern: connect → cursor → execute → fetch; always use `?`
parameterization (never f-string SQL); `row_factory = sqlite3.Row` for dict-like
rows; `executemany` + `commit` for transactional bulk loads; SQLAlchemy `engine`
+ `text()` is the portable, pandas-friendly way most pipelines connect. That completes
**Track 2**. Next: **Track 3** opens with typing and validation.